In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
import shutil
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

##Load splits and extract real images (same as ResNet_Augmentation / ablation notebooks)

BACKUP_DIR = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'
ZIP_PATH = '/content/drive/MyDrive/Thesis-work/ISIC_2019_Training_Input.zip'
UNLABELED_DIR = 'data/isic2019/ISIC_2019_Training_Input'
LABELED_DIR = 'data/labeled_real'

os.makedirs('data/isic2019', exist_ok=True)
os.makedirs(LABELED_DIR, exist_ok=True)

##Extracting full ISIC archive
if not os.path.exists(UNLABELED_DIR) or len(os.listdir(UNLABELED_DIR)) < 1000:
    print("Extracting ISIC 2019 images... (takes a few minutes)")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('data/isic2019/')
    print("Extraction complete")
else:
    print("Already extracted")

## Load Experiment A splits
train_df = pd.read_csv(f'{BACKUP_DIR}/expA_train.csv')
val_df = pd.read_csv(f'{BACKUP_DIR}/expA_val.csv')
test_df = pd.read_csv(f'{BACKUP_DIR}/expA_test.csv')
all_images = pd.concat([train_df, val_df, test_df])['image'].unique()
print(f"Total unique images needed: {len(all_images)}")

# Copying real images into labeled_real
found = 0
missing = []
for img_id in all_images:
    src = f'{UNLABELED_DIR}/{img_id}.jpg'
    dst = f'{LABELED_DIR}/{img_id}.jpg'
    if os.path.exists(dst):
        found += 1
        continue
    if os.path.exists(src):
        shutil.copy(src, dst)
        found += 1
    else:
        missing.append(img_id)

print(f"Real images available: {found}/{len(all_images)}")
if missing:
    print(f"Missing: {len(missing)} (first few: {missing[:5]})")

print("Training distribution:")
print(train_df['label'].value_counts())
print(f"\nTotal training: {len(train_df)}")
print("\nVal distribution:")
print(val_df['label'].value_counts())
print("\nTest distribution:")
print(test_df['label'].value_counts())

##Dataset / Transforms (Real Images)

Batch_Size = 16
Epochs = 30
LR = 0.001
Image_Size = 224

class SkinDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['image']
        label = self.class_to_idx[row['label']]
        img_path = f'{LABELED_DIR}/{img_id}.jpg'
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((Image_Size, Image_Size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((Image_Size, Image_Size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = SkinDataset(train_df, train_transform)
val_dataset = SkinDataset(val_df, val_transform)
test_dataset = SkinDataset(test_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=Batch_Size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=Batch_Size)
test_loader = DataLoader(test_dataset, batch_size=Batch_Size)

print(f"Classes: {train_dataset.classes}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

##Simple CNN from scratch

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SimpleCNN(len(train_dataset.classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

def train_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return accuracy_score(all_labels, all_preds), all_labels, all_preds

##Train + evaluate (run with multiple seeds for Phase-2 statistical rigor)

import random

SEEDS = [42, 123, 2024]  # running with multiple seeds;
all_runs = []

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = SimpleCNN(len(train_dataset.classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    print(f"\n{'='*40}")
    print(f"Baseline (REAL images, no SSL, no rebalancing) - seed {seed}")
    print(f"{'='*40}")

    history = {'train_acc': [], 'val_acc': []}
    for epoch in range(Epochs):
        train_loss, train_acc = train_epoch(model, train_loader)
        val_acc, _, _ = evaluate(model, val_loader)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}: Train Acc={train_acc:.3f}, Val Acc={val_acc:.3f}")

    test_acc, true_labels, pred_labels = evaluate(model, test_loader)
    report = classification_report(true_labels, pred_labels, target_names=train_dataset.classes, output_dict=True, zero_division=0)

    print(f"\nTest Acc: {test_acc:.3f} ({test_acc*100:.1f}%)")
    print(classification_report(true_labels, pred_labels, target_names=train_dataset.classes, zero_division=0))

    all_runs.append({
        'seed': seed,
        'test_acc': test_acc,
        'mel_recall': report['MEL']['recall'],
        'bkl_recall': report['BKL']['recall'],
        'nv_recall': report['NV']['recall'],
        'history': history
    })

##Summary across seeds

accs = [r['test_acc'] for r in all_runs]
mel_recalls = [r['mel_recall'] for r in all_runs]
bkl_recalls = [r['bkl_recall'] for r in all_runs]
nv_recalls = [r['nv_recall'] for r in all_runs]

print("="*40)
print("BASELINE (REAL IMAGES) - SUMMARY ACROSS SEEDS")
print("="*40)
print(f"Test Accuracy:  {np.mean(accs):.3f} +/- {np.std(accs):.3f}")
print(f"MEL Recall:     {np.mean(mel_recalls):.3f} +/- {np.std(mel_recalls):.3f}")
print(f"BKL Recall:     {np.mean(bkl_recalls):.3f} +/- {np.std(bkl_recalls):.3f}")
print(f"NV Recall:      {np.mean(nv_recalls):.3f} +/- {np.std(nv_recalls):.3f}")
print("="*40)

##Save
BACKUP_DIR = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'
os.makedirs(BACKUP_DIR_OUT, exist_ok=True)

summary = {
    'experiment': 'A',
    'data_source': 'real_isic_images',
    'note': 'Phase-2 corrected re-run of baseline using real ISIC 2019 images (data/labeled_real), replacing the Phase-1 synthetic-image pipeline-validation run',
    'runs': all_runs,
    'mean_test_acc': float(np.mean(accs)),
    'std_test_acc': float(np.std(accs)),
    'mean_mel_recall': float(np.mean(mel_recalls)),
    'std_mel_recall': float(np.std(mel_recalls)),
    'mean_bkl_recall': float(np.mean(bkl_recalls)),
    'mean_nv_recall': float(np.mean(nv_recalls))
}

output_path = f'{BACKUP_DIR}/baseline_expA_real_results.json'
with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nSaved to: {output_path}")

Mounted at /content/drive
Device: cuda
Extracting ISIC 2019 images... (takes a few minutes)
Extraction complete
Total unique images needed: 691
Real images available: 691/691
Training distribution:
label
NV     349
BKL     99
MEL     35
Name: count, dtype: int64

Total training: 483

Val distribution:
label
NV     76
BKL    21
MEL     7
Name: count, dtype: int64

Test distribution:
label
NV     75
BKL    21
MEL     8
Name: count, dtype: int64
Classes: ['BKL', 'MEL', 'NV']
Train: 483, Val: 104, Test: 104

Baseline (REAL images, no SSL, no rebalancing) - seed 42
Epoch 5: Train Acc=0.725, Val Acc=0.731
Epoch 10: Train Acc=0.725, Val Acc=0.731
Epoch 15: Train Acc=0.729, Val Acc=0.740
Epoch 20: Train Acc=0.723, Val Acc=0.731
Epoch 25: Train Acc=0.723, Val Acc=0.731
Epoch 30: Train Acc=0.720, Val Acc=0.731

Test Acc: 0.721 (72.1%)
              precision    recall  f1-score   support

         BKL       0.00      0.00      0.00        21
         MEL       0.00      0.00      0.00         8


In [ ]:
BACKUP_DIR = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'
# And save to:
output_path = f'{BACKUP_DIR}/baseline_expA_real_results.json'